# 02 · Option 1 · Fabric-native evaluation

Everything runs in Fabric: this notebook asks Copilot Chat each golden question as the test user, captures the
DAX Fabric IQ ran (workspace monitoring), scores answers with deterministic Python checks plus a judge model
deployed in Microsoft Foundry, and streams every row to the Eventhouse. The scorecard app and KQL show the results.

Optional notebook path, not the default acceptance route. Use the local setup/rehearsal scripts for the bounded demo.
Cloud authentication/Key Vault execution needs its own observed validation. No automatic jobs; live/login flags stay false.

In [ ]:
# Make the cfeval package available.
# - Local clone (VS Code / Jupyter): the cell below finds src/ in the repo.
# - Fabric notebook: uncomment the %pip line (or attach an Environment with the package).
# %pip install -q "cfeval[copilot,kusto,judge] @ git+https://github.com/bgbailey/copilot_iq_eval"
import sys, pathlib
repo = pathlib.Path.cwd()
while repo != repo.parent and not (repo / "src" / "cfeval").exists():
    repo = repo.parent
if (repo / "src" / "cfeval").exists() and str(repo / "src") not in sys.path:
    sys.path.insert(0, str(repo / "src"))
import cfeval
print("cfeval", cfeval.__version__)

In [ ]:
import os

interactive_login = False  # True only for an explicit manual sign-in; reset afterwards.
run_live = False  # Requires separate bounded approval; never enable in a published default.
max_questions = 8
outcome = None

os.environ.update({
    # Copilot Chat API: app registration + Copilot-licensed test user (docs/SETUP.md)
    "TENANT_ID": "<tenant id>",
    "TENANT_LABEL": "<tenant name / domain>",
    "OPERATOR_UPN": "eval.operator@<your-domain>",
    "AUTH_RUN_LABEL": "cfeval-option1-login",
    "CLIENT_ID": "<app registration client id>",
    "TEST_USER_UPN": "eval.tester@<your-domain>",
    "TOKEN_CACHE": "keyvault",                      # optional explicit persistence
    "KEY_VAULT_URL": "https://<your-vault>.vault.azure.net/",
    # Golden set: upload your CSV to the lakehouse Files area
    "GOLDEN_SET": "/lakehouse/default/Files/golden_questions.csv",
    # Telemetry Eventhouse
    "EVENTHOUSE_QUERY_URI": "https://<your-eventhouse>.z0.kusto.fabric.microsoft.com",
    "EVENTHOUSE_DATABASE": "<actual engine databaseName, often the KQL database GUID>",
    # DAX capture: workspace monitoring of the semantic-model workspace
    "MONITORING_QUERY_URI": "https://<monitoring-eventhouse>.z0.kusto.fabric.microsoft.com",
    "MONITORING_DATABASE": "<monitoring KQL database name>",
    "SEMANTIC_MODEL_WORKSPACE_ID": "<workspace id>",
    # Judge model in Microsoft Foundry (Azure OpenAI v1 endpoint); key kept in Key Vault
    "JUDGE_ENDPOINT": "https://<your-foundry-resource>.openai.azure.com",
    "JUDGE_DEPLOYMENT": "gpt-5-mini",
    "JUDGE_API_KEY_SECRET": "judge-api-key",
})
from cfeval.config import Settings
settings = Settings.load(env_file=None)

## Sign in as the test user (first run only)
By default this cell does nothing. For one explicit manual sign-in, set `interactive_login=True` above and keep the notebook's labelled task/service/run/tenant/expected-user context and sign-in tab in the same window. Reset the flag afterwards. Silent live execution never prompts.

In [ ]:
from cfeval.auth import DelegatedAuth, notebook_login_callback

if interactive_login or run_live:
    callback = notebook_login_callback(settings) if interactive_login else None
    auth = DelegatedAuth.from_settings(settings, login_callback=callback)
    auth.token(interactive=interactive_login)
    print("Verified test user", auth.signed_in_upn(), "tenant", settings.tenant_id)
else:
    print("No authentication or cloud calls (live/login flags false).")

## Run the golden set

In [ ]:
from cfeval.cli import build_harness

if run_live:
    from cfeval.golden import load_golden
    questions = load_golden(settings.golden_set)
    if len(questions) > max_questions or settings.repeats != 1:
        raise ValueError("Question/repeat allowance exceeded")
    outcome = build_harness(settings, option="fabric-native", questions=questions, repeats=1,
                            explain_mode="followup", chat_max_retries=0, judge_max_retries=0,
                            judge_max_calls=2 * max_questions, run_name="Optional notebook run").run()
    print(outcome.metrics)
else:
    print("No questions or judge calls (run_live=False). Use the local scripted demo for acceptance.")

In [ ]:
from pprint import pprint

cols = ["QuestionId", "Verdict", "ValueScore", "ArtifactScore", "ModelScore", "JudgeAnswerScore", "WhyScore",
        "CitedArtifacts", "QueriedModels", "LatencyMs", "DaxQueryCount", "Reasons"]
if outcome is not None:
    pprint([{key: row[key] for key in cols} for row in outcome.results])

## Look at it in the Eventhouse

In [ ]:
from cfeval.kusto import KustoQueryClient

if outcome is not None:
    kc = KustoQueryClient(settings.eventhouse_query_uri, settings=settings)
    pprint(kc.query(settings.eventhouse_database, "RunSummary(30d) | take 10"))

In [ ]:
# The ask and the DAX Fabric IQ ran for it, in time order
if outcome is not None:
    first_ask = outcome.results[0]["AskId"]
    pprint(kc.query(settings.eventhouse_database, f"AskTimeline('{first_ask}')"))

## If DAX shows as pending
Missing/unsettled DAX is incomplete evidence. Stop and diagnose; do not silently regrade or repeat a full batch.
The local scripted route's `--verify-only` resume observes publication only and never re-fetches traces.